In [1]:
!git clone --branch dev https://github.com/alpha-zero-one/llm-ai-student
%cd llm-ai-student
# check training_config.json

Cloning into 'llm-ai-student'...
remote: Enumerating objects: 263, done.
remote: Counting objects: 100% (263/263), done.
remote: Compressing objects: 100% (156/156), done.
remote: Total 263 (delta 142), reused 210 (delta 93), pack-reused 0 (from 0)
Receiving objects: 100% (263/263), 39.92 KiB | 2.35 MiB/s, done.
Resolving deltas: 100% (142/142), done.
/content/llm-ai-student


In [2]:
!pip install -r "requirements.txt"

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-573pvtwf
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-573pvtwf
  Resolved https://github.com/huggingface/transformers to commit 8ea72d12a24ed7b594bfd8e166b7705049f2a7b4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/huggingface/accelerate to /tmp/pip-req-build-28xct9yc
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/accelerate /tmp/pip-req-build-28xct9yc
  Resolved https://github.com/huggingface/accelerate to commit ae0499ea968ce0a6617abaede6997e1af498a301
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/huggingface/peft to /tmp/pip-req-build-3r2kjmhr
  Running command git c

In [3]:
!python3 "./src/single_gpu/util/download.py"

Fetching 12 files:   0% 0/12 [00:00<?, ?it/s]Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`

config.json: 100% 661/661 [00:00<00:00, 2.93MB/s]

README.md: 100% 4.07k/4.07k [00:00<00:00, 28.6MB/s]

generation_config.json: 100% 139/139 [00:00<00:00, 1.07MB/s]

merges.txt:   0% 0.00/1.67M [00:00<?, ?B/s]

.gitattributes: 100% 1.52k/1.52k [00:00<00:00, 7.12MB/s]
Fetching 12 files:   8% 1/12 [00:00<00:05,  2.06it/s]

LICENSE: 100% 7.39k/7.39k [00:00<00:00, 40.0MB/s]


model-00002-of-00002.safetensors:   0% 0.00/1.21G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   2% 21.0M/1.2

In [ ]:
#upload the training and evaluationset
#upload weights

In [4]:
import shutil
import os

if os.path.exists("./local/download/sets.zip"):
    shutil.unpack_archive("./local/download/sets.zip", "./local/download", 'zip')

In [5]:
import shutil
import os

if os.path.exists("./local/trained.zip"):
    shutil.unpack_archive("./local/trained.zip", "./local", 'zip')

In [6]:
!python3 "./src/single_gpu/training/training_pipeline.py"

Progress: 1/2
Start: 2025-05-12 15:24:58
End: 2025-05-12 15:27:07
Progress: 2/2
Start: 2025-05-12 15:27:07
End: 2025-05-12 15:29:02


In [7]:
!python3 "./src/single_gpu/evaluate/evaluation_pipeline.py"

Progress: 1/1
Start: 2025-05-12 15:32:40
End: 2025-05-12 15:36:51
Progress: 1/2
Start: 2025-05-12 15:36:51
End: 2025-05-12 15:43:31
Progress: 2/2
Start: 2025-05-12 15:43:31
End: 2025-05-12 15:50:05


In [8]:
!python3 "./src/single_gpu/visualize/visualize.py"

In [9]:
import os
import json
import zipfile

root_folder = './local/trained'
subpaths = []

with open("./config/training_config.json", 'r') as file:
    training_config = json.load(file)
    for base_dataset in training_config["base_dataset"]:
        for base_model in training_config["base_model"]:
            for lora_dropout in training_config["lora_dropout"]:
                for lora_rank_alpha in training_config["lora_rank_alpha"]:
                    lora_rank = lora_rank_alpha["lora_rank"]
                    lora_alpha = lora_rank_alpha["lora_alpha"]
                    for learning_rate in training_config["learning_rate"]:
                        num_epochs = training_config["num_epochs"]
                        trained_path = (f"{base_model}/{base_dataset}/"
                                        f"lora_rank/{lora_rank}/lora_alpha/{lora_alpha}/lora_dropout/{lora_dropout}/"
                                        f"learning_rate/{learning_rate}/num_epochs/{num_epochs}/model")
                        subpaths.append(trained_path)

with zipfile.ZipFile(f'{root_folder}.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for subpath in subpaths:
        abs_path = os.path.join(root_folder, subpath)
        if os.path.exists(abs_path):
            for foldername, subfolders, filenames in os.walk(abs_path):
                for filename in filenames:
                    abs_file = os.path.join(foldername, filename)
                    rel_path = os.path.relpath(abs_file, os.path.dirname(root_folder))
                    zipf.write(abs_file, rel_path)


In [ ]:
#download summary.zip
#download model weights